In [6]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import class_weight

In [11]:
import cv2
import numpy as np
import tensorflow as tf

# Load the trained model
model = tf.keras.models.load_model('resnet50_model.h5')

# Load label encoder if needed
label_encoder = LabelEncoder()
# Assume you have saved label mappings previously, otherwise train it again
label_encoder.classes_ = np.load('flattened_features.npy', allow_pickle=True)

# Load test image
test_image_path = "Screenshot 2022-11-29 at 2.28.56 PM.png"  # Update path
test_image = cv2.imread(test_image_path)
gray = cv2.cvtColor(test_image, cv2.COLOR_BGR2GRAY)

# Initialize SIFT detector
sift = cv2.SIFT_create()

# Detect keypoints and compute descriptors
keypoints, descriptors = sift.detectAndCompute(gray, None)

# Check if descriptors exist
if descriptors is None:
    raise ValueError("No SIFT features detected in the test image.")

# Flatten the descriptors (reshape if needed)
test_feature = descriptors.flatten().reshape(1, -1)  # Reshape for model input

# Ensure the correct input shape
expected_feature_size = model.input_shape[1]  # Expected feature length
if test_feature.shape[1] > expected_feature_size:
    test_feature = test_feature[:, :expected_feature_size]  # Trim excess features
elif test_feature.shape[1] < expected_feature_size:
    padding = np.zeros((1, expected_feature_size - test_feature.shape[1]))
    test_feature = np.hstack((test_feature, padding))  # Pad with zeros



prediction = model.predict(test_feature)  # Get probability distribution
predicted_label = np.argmax(prediction, axis=1)[0]  # Extract the highest probability class

# Decode the predicted label into a location name
predicted_location = label_encoder.inverse_transform([predicted_label])[0]

print(f"Predicted Location: {predicted_location}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
Predicted Location: [ 28.14904    17.458422   13.69403    18.26695    38.81706    15.651173
  10.7353945  14.28017    67.528145   27.191898   16.586994   20.163113
  36.065033   15.703625   11.59339    23.150959   70.11663    24.976545
  13.051172   14.565672   24.861406   15.489339   14.97356    29.681664
  28.34627    14.141151    9.940298   12.2884865  24.763327   17.832409
  15.673347   18.740725   35.97612    18.926012   15.364606   25.257568
  49.20448    20.480383   11.508742   15.565458   92.66396    28.281664
  18.01066    31.379318   50.706398   19.17079    11.384861   29.084648
  99.91407    33.148186   14.554158   18.230703   31.679531   21.779957
  17.002346   30.65501    36.829212   16.834116   12.239446   17.317911
  32.03774    22.423454   16.411087   18.852879   35.340725   16.798721
  11.938806   20.803198   48.6968     25.292751   15.419616   18.196375
  92.58337    29.073133   11.358422   19.420683   49.778465   30.853518
  18.2